[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riccardoberta/regressione/blob/main/04_generalizzazione_overfitting.ipynb)

# Quando il modello si illude di essere bravo

Un modello può sembrare bravissimo sui dati che ha già visto. Ma il vero obiettivo del machine learning è prevedere bene su casi nuovi. In questo episodio introduciamo training set, test set, generalizzazione e overfitting.

> **Missione** — Separare i dati in allenamento e test, confrontare modelli diversi e capire quando un modello sta imparando una regola oppure sta memorizzando i dati.


In [ ]:
# Setup: imports e download del dataset (solo la prima volta)
import urllib.request, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_URL = (
    "https://www.dropbox.com/scl/fi/0l5n46qjwcd5moj3w7d8p/"
    "fifa.zip?rlkey=rcqhagvq5ttlvna5t5r3vn1bm&st=uzplzs5o&dl=1"
)
ZIP_PATH = Path("fifa.zip")
DATA_DIR = Path("fifa_data")

if not ZIP_PATH.exists():
    print("Scarico il dataset...")
    urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

if not list(DATA_DIR.rglob("*.csv")):
    DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

csv_files = (
    list(DATA_DIR.rglob("players_22.csv"))
    or list(DATA_DIR.rglob("*players*.csv"))
)
raw = pd.read_csv(csv_files[0], low_memory=False)
print(f"Caricato: {raw.shape[0]} giocatori, {raw.shape[1]} colonne")
raw.head()


In [ ]:
# Creiamo una versione didattica piccola e pulita
wanted_columns = [
    "short_name", "age", "overall", "potential", "wage_eur", "value_eur",
    "club_name", "league_name", "player_positions"
]

available = [c for c in wanted_columns if c in raw.columns]
df = raw[available].copy()

# Manteniamo solo i giocatori con valore noto e positivo.
df = df.dropna(subset=["value_eur", "age", "overall", "potential"])
df = df[df["value_eur"] > 0]

# Riduciamo l'effetto dei super-outlier per i primi grafici didattici.
df = df[df["value_eur"] <= df["value_eur"].quantile(0.99)]

# Per leggibilita' useremo spesso il valore in milioni di euro.
df["value_mln_eur"] = df["value_eur"] / 1_000_000
if "wage_eur" in df.columns:
    df["wage_k_eur"] = df["wage_eur"] / 1_000

print("Forma del dataset didattico:", df.shape)
df.head(10)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
)
from sklearn.model_selection import train_test_split

> **Generalizzazione** — Un modello generalizza quando funziona bene su dati nuovi, non solo sui dati usati per addestrarlo. Questo e' il cuore del machine learning: non vogliamo una memoria dei casi passati, vogliamo una regola utile per il futuro.


### Teoria — Errore di training vs errore atteso

Quello che vorremmo davvero misurare è quanto sbaglierà il modello sui **casi futuri**, cioè l'**errore atteso**:

$\displaystyle R(f) \;=\; \mathbb{E}_{(x, y)}\!\left[(y - f(x))^2\right]$

Non possiamo calcolarlo esattamente, perché non conosciamo tutti i casi del futuro. Possiamo però calcolare l'**errore empirico** sui dati che abbiamo:

$\displaystyle \hat{R}_{\text{train}}(f) \;=\; \frac{1}{n_{\text{train}}} \sum_{i \in \text{train}} (y_i - f(x_i))^2$

Il punto delicato: $\hat{R}_{\text{train}}$ può ingannarci. Un modello molto flessibile può renderlo quasi zero "imparando a memoria" il training set, restando però pessimo sui dati nuovi. Per stimare onestamente $R(f)$ misuriamo l'errore su un **test set** mai visto durante l'addestramento:

$\displaystyle \hat{R}_{\text{test}}(f) \;=\; \frac{1}{n_{\text{test}}} \sum_{i \in \text{test}} (y_i - f(x_i))^2$

È questo numero che dobbiamo guardare per giudicare se il modello generalizza.


## Dividiamo i dati: training e test

In [ ]:
features = [
    c for c in ["age", "overall", "potential", "wage_eur"]
    if c in df.columns
]
X = df[features]
y = df["value_mln_eur"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

print("Esempi di training:", len(X_train))
print("Esempi di test:", len(X_test))

> **Training set e test set** — Il training set serve per imparare i parametri del modello. Il test set resta nascosto fino alla valutazione finale. Se il modello va bene anche sul test set, abbiamo piu' fiducia che abbia imparato una regola generale.


## Modello lineare valutato correttamente

In [ ]:
lin = LinearRegression()
lin.fit(X_train, y_train)

pred_train_lin = lin.predict(X_train)
pred_test_lin = lin.predict(X_test)

def metrics(y_true, y_pred):
    return {
        "MAE_mln": mean_absolute_error(y_true, y_pred),
        "RMSE_mln": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred)
    }

results = []
results.append({
    "modello": "Lineare", "dati": "training",
    **metrics(y_train, pred_train_lin),
})
results.append({
    "modello": "Lineare", "dati": "test",
    **metrics(y_test, pred_test_lin),
})
pd.DataFrame(results).round(3)

## Un modello molto flessibile: albero decisionale

In [ ]:
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

pred_train_tree = tree.predict(X_train)
pred_test_tree = tree.predict(X_test)

results.append({
    "modello": "Albero non limitato", "dati": "training",
    **metrics(y_train, pred_train_tree),
})
results.append({
    "modello": "Albero non limitato", "dati": "test",
    **metrics(y_test, pred_test_tree),
})
pd.DataFrame(results).round(3)

> **Overfitting** — Overfitting significa che il modello si adatta troppo ai dettagli del training set. In quel caso l'errore sul training puo' diventare molto basso, ma l'errore sul test resta alto o peggiora. E' come uno studente che impara a memoria gli esercizi svolti ma non sa risolverne uno nuovo.


## Limitiamo la complessità dell'albero

In [ ]:
tree_small = DecisionTreeRegressor(max_depth=4, random_state=42)
tree_small.fit(X_train, y_train)

pred_train_small = tree_small.predict(X_train)
pred_test_small = tree_small.predict(X_test)

results.append({
    "modello": "Albero max_depth=4", "dati": "training",
    **metrics(y_train, pred_train_small),
})
results.append({
    "modello": "Albero max_depth=4", "dati": "test",
    **metrics(y_test, pred_test_small),
})

pd.DataFrame(results).round(3)

> **Complessita' del modello** — Un modello troppo semplice puo' non catturare la struttura dei dati: underfitting. Un modello troppo complesso puo' catturare anche rumore e casi particolari: overfitting. L'obiettivo e' trovare un equilibrio che funzioni bene sui dati nuovi.


## Vediamo l'overfitting come una curva


### Teoria — La curva a U dell'overfitting

Aumentando la complessità del modello (per esempio la profondità massima dell'albero) succede tipicamente che:

- L'**errore di training** cala sempre: più il modello è flessibile, più si adatta ai dati visti.
- L'**errore di test** prima cala (il modello impara la regola vera), poi **risale** (il modello impara il rumore del training).

Il punto migliore è il **minimo della curva di test**. È il classico equilibrio tra:

- **Bias** (errore sistematico): un modello troppo semplice non riesce a catturare il vero pattern — *underfitting*.
- **Varianza**: un modello troppo flessibile cambia molto al variare dei dati di training — *overfitting*.


In [ ]:
# Errore su training e test al variare della profondita'
profondita = list(range(1, 21))
err_train = []
err_test = []

for d in profondita:
    m = DecisionTreeRegressor(max_depth=d, random_state=42)
    m.fit(X_train, y_train)
    err_train.append(mean_absolute_error(y_train, m.predict(X_train)))
    err_test.append(mean_absolute_error(y_test, m.predict(X_test)))

plt.figure(figsize=(8,5))
plt.plot(
    profondita, err_train, "o-",
    label="Errore su training", color="#2a9d8f",
)
plt.plot(
    profondita, err_test, "o-",
    label="Errore su test", color="#e76f51",
)
plt.xlabel("Profondita' massima dell'albero")
plt.ylabel("MAE [milioni di euro]")
plt.title("Underfitting vs overfitting al crescere della complessita'")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

best_d = profondita[int(np.argmin(err_test))]
print(f"Profondita' con errore di test minimo: {best_d}")


## Previsioni finali su casi di test

In [ ]:
test_view = X_test.copy()
test_view["valore_reale"] = y_test
test_view["lineare"] = pred_test_lin
test_view["albero_non_limitato"] = pred_test_tree
test_view["albero_limitato"] = pred_test_small

# Recuperiamo i nomi dei giocatori corrispondenti
names = df.loc[
    test_view.index,
    ["short_name", "club_name", "player_positions"],
]
final_view = pd.concat([names, test_view], axis=1)
final_view.sample(12, random_state=5).round(2)

---

> **Discussione finale** — Quale modello scegliereste per aiutare davvero una societa' sportiva? Non basta dire quello con errore minore sul training. Bisogna guardare il test set e ragionare sulla stabilita' del modello.


---

> **Cosa abbiamo imparato nella serie** — Abbiamo trasformato una domanda concreta, quanto vale un calciatore?, in un problema di regressione. Abbiamo esplorato dati, costruito un primo modello, aggiunto variabili, misurato errori e infine distinto tra prestazione apparente e generalizzazione.
